In [1]:
import pandas as pd
from pathlib import Path

base_dir = Path.cwd().parent
processed_dir = base_dir / "data/processed"

dfs = {}

for file in processed_dir.glob("*.csv"):
    name = file.stem
    dfs[name] = pd.read_csv(file)

In [2]:
orders = dfs["orders_clean"]
customers = dfs["customers_clean"]
geolocation = dfs["geolocation_clean"]
sellers = dfs["sellers_clean"]
products = dfs["products_clean"]
category_name_translation = dfs["category_name_translation_clean"]
order_items = dfs["order_items_clean"]
order_payments = dfs["order_payments_clean"]
order_reviews = dfs["order_reviews_clean"]

# Data Model Classification

```
| Table                     | Role                 | Granularity                          |
|---------------------------|----------------------|--------------------------------------|
| order_items               | Fact table           | 1 riga = 1 articolo venduto          |
| orders                    | Dimension table      | 1 riga = 1 ordine                    |
| customers                 | Dimension table      | 1 riga = 1 customer_id               |
| products                  | Dimension table      | 1 riga = 1 prodotto                  |
| sellers                   | Dimension table      | 1 riga = 1 venditore                 |
| geolocation               | Lookup table         | 1 riga = 1 osservazione per ZIP code |
| order_payments            | Secondary fact table | 1 riga = 1 transazione di pagamento  |
| order_reviews             | Secondary fact table | 1 riga = 1 recensione per ordine     |
| category_name_translation | lookup table         | 1 riga = 1 categoria                 |
```

# Data Model Relationships

## orders -> order_items
* orders PK: order_id
* order_items FK order_id
* 1:N
* un ordine può avere più articoli venduti ma per ogni articolo venduto c'è solo un ordine

## products -> order_items
* products PK: product_id
* order_items FK: product_id
* 1:N
* un prodotto può essere contenuto in più articoli venduti ma per ogni articolo venduto c'è solo un prodotto

## sellers -> order_items
* sellers PK: seller_id
* order_items FK: seller_id
* 1:N
* un venditore può vendere più articoli ma per ogni articolo venduto c'è solo un venditore

## customers -> orders
* customers PK: customer_id
* orders FK: customer_id
* 1:1
* ad ogni customer_id corrisponde un solo ordine e viceversa

## orders -> order_payments
* orders PK: order_id
* order_payments FK: order_id
* 1:N
* un ordine può avere più transazioni di pagamento ma ogni transazione di pagamento si riferisce ad un solo ordine

## orders -> order_reviews
* orders PK: order_id
* order_reviews FK: order_id
* 1:N
* un ordine può avere più recensioni ed ogni recensione è riferita ad un solo ordine

## geolocation <-> customers
* collegamento logico tramite ZIP code prefix
* relazione di lookup geografico
* 1:N

## geolocation <-> sellers
* collegamento logico tramite ZIP code prefix
* relazione di lookup geografico
* 1:N

## category_name_translation <-> products
* collegamento logico tramite product_category_name
* relazione di lookup linguistico
* 1:N

# Data Model Validation

## Referential Integrity Validation

Non sono emerse violazioni dell'integrità referenziale in quanto tutte le FK delle tabelle figlie trovano corrispondenza nelle rispettive PK delle tabelle padre.

Le seguenti relazioni di lookup geografico e linguistico non sono state validate in quanto si tratta di collegamenti logici di arricchimento e non vincoli strutturali del modello relazionale:

* geolocation <-> customers
* geolocation <-> sellers
* category_name_translation <-> products

In [15]:
(
    ~order_items["order_id"].isin(orders["order_id"])
).sum()

np.int64(0)

In [17]:
(
    ~order_items["product_id"].isin(products["product_id"])
).sum()

np.int64(0)

In [20]:
(
    ~order_items["seller_id"].isin(sellers["seller_id"])
).sum()

np.int64(0)

In [24]:
(
    ~orders["customer_id"].isin(customers["customer_id"])
).sum()

np.int64(0)

In [25]:
(
    ~order_payments["order_id"].isin(orders["order_id"])
).sum()

np.int64(0)

In [40]:
(
    ~order_reviews["order_id"].isin(orders["order_id"])
).sum()

np.int64(0)

## Cardinality Validation

Dalla validazione delle cardinalità è emerso che tutte le relazioni strutturali ipotizzate nel data model relationships sono confermate, ad eccezione di *customers* e *orders*. Queste hanno una relazione 1:1, e non 1:N come ipotizzato precedentemente, in quanto ad ogni ordine viene assegnato un *customer_id* univoco, anche se si tratta dello stesso cliente.

Inoltre, è bene precisare che la validazione ha confermato una cardinalità 1:N tra *orders* e *order_reviews*, in quanto esistono:

* 98126 ordini con 1 recensione
* 543 ordini con 2 recensioni
* 4 ordini con 3 recensioni

In [71]:
order_items.groupby(
    "order_id"
).size().value_counts().sort_index()

1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
20        2
21        1
Name: count, dtype: int64

In [73]:
order_items.groupby(
    "product_id"
).size().value_counts().sort_index()

1      18117
2       5817
3       2651
4       1534
5        994
       ...  
388        1
392        1
484        1
488        1
527        1
Name: count, Length: 138, dtype: int64

In [74]:
order_items.groupby(
    "seller_id"
).size().value_counts().sort_index()

1       509
2       328
3       212
4       152
5       142
       ... 
1551      1
1775      1
1931      1
1987      1
2033      1
Name: count, Length: 257, dtype: int64

In [112]:
orders.groupby(
    "customer_id"
).size().value_counts().sort_index()

1    99441
Name: count, dtype: int64

In [77]:
order_payments.groupby(
    "order_id"
).size().value_counts().sort_index()

1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

In [80]:
order_reviews.groupby(
    "order_id"
).size().value_counts().sort_index()

1    98126
2      543
3        4
Name: count, dtype: int64

## Join Validation

La validazione delle join tra le tabelle con relazioni strutturali non ha evidenziato duplicazioni inattese o perdita di righe.

In [104]:
order_items.shape[0],\
orders.merge(order_items, on="order_id").shape[0],\
products.merge(order_items, on="product_id").shape[0],\
sellers.merge(order_items, on="seller_id").shape[0]

(112650, 112650, 112650, 112650)

In [106]:
orders.shape[0],\
customers.merge(orders, on="customer_id").shape[0]

(99441, 99441)

In [108]:
order_payments.shape[0],\
orders.merge(order_payments, on="order_id").shape[0]

(103886, 103886)

In [111]:
order_reviews.shape[0],\
orders.merge(order_reviews, on="order_id").shape[0]

(99224, 99224)